# Estudo Dirigido 9 – Extração de Características por Momentos
**ENGG64 – Visão Computacional · UFBA · Semestre 2026/2**

Cada seção abaixo corresponde a uma etapa do enunciado. Execute as células **em ordem**.

- **Objeto analisado:** coração (silhueta branca sobre fundo preto), arquivo `coracao.png`.
- **Convenção:** $u$ = coluna (eixo horizontal) e $v$ = linha (eixo vertical, crescendo para baixo), com origem no canto superior esquerdo.

## 0. Configuração
Importa as bibliotecas, define cores e uma função auxiliar para exibir a máscara com zoom.

In [ ]:
import os, json
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

os.makedirs("figuras", exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})

COR = dict(contorno="#39ff14", centroide="#ff00c8", eixo1="#ff2d2d",
           eixo2="#1e90ff", elipse="#ff9f0a")

def mostrar_mascara(ax, M, margem=40):
    """Mostra uma imagem binária com zoom em torno do objeto (coordenadas originais)."""
    ys, xs = np.nonzero(M)
    ax.imshow(M, cmap="gray", vmin=0, vmax=1)
    ax.set_xlim(xs.min() - margem, xs.max() + margem)
    ax.set_ylim(ys.max() + margem, ys.min() - margem)
    ax.set_xlabel("u (pixels)")
    ax.set_ylabel("v (pixels)")

## 1. Aquisição da imagem
No Google Colab, a célula pede o upload de `coracao.png` (se o arquivo ainda não estiver na pasta de trabalho).

In [ ]:
NOME_IMG = "coracao.png"
try:
    from google.colab import files
    if not os.path.exists(NOME_IMG):
        print("Faça o upload da imagem original (coracao.png):")
        NOME_IMG = list(files.upload().keys())[0]
except ImportError:
    pass  # fora do Colab: usa o arquivo local

img_bgr = cv2.imread(NOME_IMG, cv2.IMREAD_COLOR)
assert img_bgr is not None, "Não foi possível abrir a imagem."
H, W = img_bgr.shape[:2]
print(f"Imagem: {W} x {H} pixels")

plt.figure(figsize=(6, 4.2))
plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
plt.title("Imagem original")
plt.axis("off")
plt.savefig("figuras/fig01_original.png", dpi=150, bbox_inches="tight")
plt.show()

## 2. Obtenção da imagem binária
Objeto claro sobre fundo escuro: usa-se **limiarização de Otsu** na imagem em tons de cinza.

$$B(u,v)=\begin{cases}1,&\text{pixel da região de interesse}\\0,&\text{caso contrário}\end{cases}$$

In [ ]:
cinza = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
limiar, bin255 = cv2.threshold(cinza, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
B = (bin255 > 0).astype(np.uint8)          # B(u,v) em {0, 1}

print(f"Limiar de Otsu: {limiar:.0f}")
print("Valores presentes em B:", np.unique(B))

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); ax[0].set_title("1. Imagem original")
ax[1].imshow(B, cmap="gray", vmin=0, vmax=1);           ax[1].set_title("2. Imagem binária B(u,v)")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig("figuras/fig02_original_binaria.png", dpi=150)
plt.show()

## 3. Seleção da região de interesse
Identifica os componentes conexos e mantém apenas o objeto a ser analisado (o maior componente).

In [ ]:
n_comp, rotulos, stats, _ = cv2.connectedComponentsWithStats(B, connectivity=8)
print("Componentes conexos (sem contar o fundo):", n_comp - 1)

idx = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])      # maior componente
mask = (rotulos == idx).astype(np.uint8)              # máscara só com o objeto
n_pixels = int(mask.sum())

x0, y0, w, h = stats[idx, :4]
print(f"Caixa delimitadora: u de {x0} a {x0 + w - 1}, v de {y0} a {y0 + h - 1}  ({w} x {h} px)")
print(f"Número de pixels da região: {n_pixels}")

fig, ax = plt.subplots(figsize=(6, 5))
mostrar_mascara(ax, mask)
ax.set_title(f"Máscara da região selecionada ({n_pixels} pixels)")
plt.tight_layout()
plt.savefig("figuras/fig03_mascara.png", dpi=150)
plt.show()

## 4. Momento de ordem zero (área)
$$m_{pq}=\sum_u\sum_v u^p\,v^q\,B(u,v),\qquad m_{00}=\sum_u\sum_v B(u,v)=A$$

In [ ]:
v, u = np.indices(mask.shape)                 # grades de coordenadas: u = coluna, v = linha
u = u.astype(np.float64)
v = v.astype(np.float64)

def momento(Bin, p, q):
    """Momento geométrico m_pq de uma imagem binária."""
    return float(np.sum((u ** p) * (v ** q) * Bin))

m00 = momento(mask, 0, 0)
A = m00
print(f"m00 = {m00:.0f}  ->  área A = {A:.0f} pixels")
print(f"Contagem direta de pixels da máscara = {n_pixels}")
print("Valores iguais?", m00 == n_pixels)

## 5. Centroide
$$u_c=\frac{m_{10}}{m_{00}},\qquad v_c=\frac{m_{01}}{m_{00}}$$

In [ ]:
m10 = momento(mask, 1, 0)
m01 = momento(mask, 0, 1)
uc = m10 / m00
vc = m01 / m00

print(f"m10 = {m10:.0f}")
print(f"m01 = {m01:.0f}")
print(f"Centroide: (uc, vc) = ({uc:.3f}, {vc:.3f})")

fig, ax = plt.subplots(figsize=(6, 5))
mostrar_mascara(ax, mask)
ax.plot(uc, vc, "+", color=COR["centroide"], ms=20, mew=3, label="centroide")
ax.set_title(f"Centroide  (uc, vc) = ({uc:.2f}, {vc:.2f})")
ax.legend(loc="upper right")
plt.tight_layout()
plt.savefig("figuras/fig04_centroide.png", dpi=150)
plt.show()

## 6. Momentos centrais de segunda ordem
$$\mu_{pq}=\sum_u\sum_v (u-u_c)^p\,(v-v_c)^q\,B(u,v),\qquad
\mathbf{J}=\begin{bmatrix}\mu_{20}&\mu_{11}\\ \mu_{11}&\mu_{02}\end{bmatrix}$$

In [ ]:
def momento_central(Bin, p, q, uc, vc):
    return float(np.sum(((u - uc) ** p) * ((v - vc) ** q) * Bin))

mu20 = momento_central(mask, 2, 0, uc, vc)
mu02 = momento_central(mask, 0, 2, uc, vc)
mu11 = momento_central(mask, 1, 1, uc, vc)
J = np.array([[mu20, mu11],
              [mu11, mu02]])

print(f"mu20 = {mu20:.4f}")
print(f"mu02 = {mu02:.4f}")
print(f"mu11 = {mu11:.4f}")
print("\nMatriz J =")
print(J)

# Conferência com a implementação do OpenCV
mc = cv2.moments(mask, binaryImage=True)
print("\nConferência com cv2.moments:")
for nome, meu in [("m00", m00), ("m10", m10), ("m01", m01), ("mu20", mu20), ("mu02", mu02), ("mu11", mu11)]:
    print(f"  {nome}: meu = {meu:.4f} | OpenCV = {mc[nome]:.4f}")

## 7. Autovalores e autovetores de $\mathbf{J}$
$\mathbf{J}$ é simétrica; usa-se `np.linalg.eigh`. Os autovalores são ordenados do maior ($\lambda_1$, eixo principal) para o menor ($\lambda_2$, eixo secundário).
O sinal de um autovetor é arbitrário, então padroniza-se $q_u \ge 0$.

In [ ]:
def autosistema(J):
    w, V = np.linalg.eigh(J)
    ordem = np.argsort(w)[::-1]                  # maior -> menor
    lam1, lam2 = w[ordem]
    q1, q2 = V[:, ordem[0]], V[:, ordem[1]]
    if q1[0] < 0 or (q1[0] == 0 and q1[1] < 0): q1 = -q1
    if q2[0] < 0 or (q2[0] == 0 and q2[1] < 0): q2 = -q2
    return lam1, lam2, q1, q2

lam1, lam2, q1, q2 = autosistema(J)

print(f"Autovalor principal   lambda1 = {lam1:.4f}")
print(f"Autovalor secundário  lambda2 = {lam2:.4f}")
print(f"Autovetor do eixo principal   q1 = [{q1[0]:.6f}, {q1[1]:.6f}]")
print(f"Autovetor do eixo secundário  q2 = [{q2[0]:.6f}, {q2[1]:.6f}]")

## 8. Orientação principal
$$\theta=\operatorname{atan2}(q_v,q_u),\qquad \theta_{\text{graus}}=\theta\,\frac{180}{\pi}$$

Como $v$ cresce para baixo, ângulos positivos correspondem a rotação **no sentido horário** na tela. O eixo tem ambiguidade de $180^\circ$ ($\mathbf{q}$ e $-\mathbf{q}$ são o mesmo eixo).

In [ ]:
theta = np.arctan2(q1[1], q1[0])
theta_graus = np.degrees(theta)
print(f"theta = {theta:.6f} rad = {theta_graus:.4f} graus")

# comprimentos usados só para desenhar os eixos (justificativa na etapa 9)
a = 2 * np.sqrt(lam1 / m00)
b = 2 * np.sqrt(lam2 / m00)

def desenhar_analise(ax, M, uc, vc, q1, q2, a, b, theta_graus, contorno=True, elipse=True):
    """Sobrepõe contorno, elipse equivalente, eixos e centroide em um eixo do matplotlib."""
    if contorno:
        cs, _ = cv2.findContours(M, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        c = max(cs, key=cv2.contourArea)[:, 0, :]
        ax.plot(c[:, 0], c[:, 1], color=COR["contorno"], lw=1.5, label="contorno")
    if elipse:
        ax.add_patch(Ellipse((uc, vc), 2 * a, 2 * b, angle=theta_graus, fill=False,
                             ec=COR["elipse"], lw=2.2, ls="--", label="elipse equivalente"))
    ax.plot([uc - a * q1[0], uc + a * q1[0]], [vc - a * q1[1], vc + a * q1[1]],
            color=COR["eixo1"], lw=2, label="eixo principal")
    ax.plot([uc - b * q2[0], uc + b * q2[0]], [vc - b * q2[1], vc + b * q2[1]],
            color=COR["eixo2"], lw=2, label="eixo secundário")
    ax.plot(uc, vc, "o", color=COR["centroide"], ms=7, mec="white", label="centroide")

fig, ax = plt.subplots(figsize=(8, 5))
mostrar_mascara(ax, mask)
desenhar_analise(ax, mask, uc, vc, q1, q2, a, b, theta_graus, contorno=False, elipse=False)
ax.set_title(f"Centroide e eixos principais  (θ = {theta_graus:.2f}°)")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.savefig("figuras/fig05_eixos.png", dpi=150)
plt.show()

## 9. Elipse equivalente
Para uma elipse sólida de semi-eixos $a$ e $b$, a variância dos pixels ao longo de cada eixo vale $a^2/4$ e $b^2/4$. Como $\lambda_i$ é uma soma (não média), divide-se por $m_{00}$:

$$a=2\sqrt{\frac{\lambda_1}{m_{00}}},\qquad b=2\sqrt{\frac{\lambda_2}{m_{00}}}$$

A elipse é centrada em $(u_c,v_c)$ e girada de $\theta$.

In [ ]:
a = 2 * np.sqrt(lam1 / m00)       # semi-eixo maior
b = 2 * np.sqrt(lam2 / m00)       # semi-eixo menor
print(f"Semi-eixo principal  a = {a:.3f} px")
print(f"Semi-eixo secundário b = {b:.3f} px")
print(f"Área da elipse = pi*a*b = {np.pi * a * b:.1f} px²  (área da região = {A:.0f} px²)")

fig, ax = plt.subplots(figsize=(8, 5))
mostrar_mascara(ax, mask)
desenhar_analise(ax, mask, uc, vc, q1, q2, a, b, theta_graus)
ax.set_title("Elipse equivalente, eixos, centroide e contorno")
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5))
plt.tight_layout()
plt.savefig("figuras/fig06_elipse.png", dpi=150)
plt.show()

## 10. Descritor de alongamento
$$\rho=\sqrt{\frac{\lambda_{\max}}{\lambda_{\min}}}$$
$\rho\approx1$: região aproximadamente circular · $\rho>1$: região alongada.

In [ ]:
rho = np.sqrt(lam1 / lam2)
print(f"rho = {rho:.4f}")
print("Interpretação:", "região aproximadamente circular (rho ≈ 1)" if rho < 1.1
      else "região alongada (rho > 1)")

## 11. Influência da translação
Desloca o mesmo objeto (sem alterar forma ou orientação) e recalcula todas as características.
Uma função `calcular_tudo` repete as etapas 4–10 para qualquer máscara.

In [ ]:
def calcular_tudo(M):
    m00_ = momento(M, 0, 0); m10_ = momento(M, 1, 0); m01_ = momento(M, 0, 1)
    uc_, vc_ = m10_ / m00_, m01_ / m00_
    mu20_ = momento_central(M, 2, 0, uc_, vc_)
    mu02_ = momento_central(M, 0, 2, uc_, vc_)
    mu11_ = momento_central(M, 1, 1, uc_, vc_)
    l1, l2, p1, p2 = autosistema(np.array([[mu20_, mu11_], [mu11_, mu02_]]))
    th = np.degrees(np.arctan2(p1[1], p1[0]))
    return dict(m00=m00_, m10=m10_, m01=m01_, uc=uc_, vc=vc_,
                mu20=mu20_, mu02=mu02_, mu11=mu11_, lam1=l1, lam2=l2,
                theta_graus=th, rho=np.sqrt(l1 / l2),
                a=2 * np.sqrt(l1 / m00_), b=2 * np.sqrt(l2 / m00_), q1=p1, q2=p2)

dx, dy = 200, 120                                      # deslocamento em pixels
T = np.float32([[1, 0, dx], [0, 1, dy]])
mask2 = cv2.warpAffine(mask, T, (W, H), flags=cv2.INTER_NEAREST)
assert mask2.sum() == mask.sum(), "O objeto saiu do quadro: reduza o deslocamento."

r1 = calcular_tudo(mask)       # antes
r2 = calcular_tudo(mask2)      # depois
assert np.isclose(r1["theta_graus"], theta_graus) and np.isclose(r1["mu20"], mu20)  # coerência com as etapas anteriores
print(f"Deslocamento aplicado: (dx, dy) = ({dx}, {dy}) pixels")

In [ ]:
chaves = ["m00", "m10", "m01", "uc", "vc", "mu20", "mu02", "mu11", "lam1", "lam2", "theta_graus", "rho"]
print(f"{'grandeza':<13}{'antes':>16}{'depois':>16}{'variação':>14}")
for k in chaves:
    print(f"{k:<13}{r1[k]:>16.4f}{r2[k]:>16.4f}{r2[k] - r1[k]:>14.4f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 4.3))
for eixo, M, r, tit in [(ax[0], mask, r1, "Antes do deslocamento"),
                        (ax[1], mask2, r2, f"Depois (dx={dx}, dy={dy})")]:
    eixo.imshow(M, cmap="gray", vmin=0, vmax=1)
    desenhar_analise(eixo, M, r["uc"], r["vc"], r["q1"], r["q2"], r["a"], r["b"], r["theta_graus"])
    eixo.set_title(f"{tit}\n(uc, vc) = ({r['uc']:.1f}, {r['vc']:.1f})")
    eixo.set_xlabel("u (pixels)"); eixo.set_ylabel("v (pixels)")
plt.tight_layout()
plt.savefig("figuras/fig07_translacao.png", dpi=150)
plt.show()

## Salvando os resultados
Gera `resultados.json` e `figuras.zip` (no Colab, o download começa automaticamente).

In [ ]:
import shutil

def para_json(d):
    return {k: (x.tolist() if hasattr(x, "tolist") else float(x)) for k, x in d.items()}

resultados = {"antes": para_json(r1), "depois": para_json(r2),
              "deslocamento_px": [dx, dy], "limiar_otsu": float(limiar)}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

shutil.make_archive("figuras", "zip", "figuras")
try:
    from google.colab import files
    files.download("figuras.zip")
    files.download("resultados.json")
except ImportError:
    print("Arquivos salvos: figuras.zip e resultados.json")